# Cello hand-tracking benchmark: MediaPipe vs RTMW

Runs two hand-tracking models on your clips and compares how often each finds the hand while the picture is slow, medium and fast moving.

**Before you start:** Runtime, Change runtime type, choose a GPU if offered (optional). Then run the cells in order. Your clips stay in this temporary session and are deleted when it closes. Do not upload footage of people without their consent.

*This notebook has not been run by its author. If a cell fails, copy the error message back into the chat.*

## 1. Set up
MediaPipe 0.10.14 is pinned because it bundles its models in the package. It needs Python 3.12, so we create a small separate Python 3.12 environment.

In [ ]:
!pip -q install uv
!uv venv --python 3.12 /content/venv
!uv pip install --python /content/venv/bin/python "mediapipe==0.10.14" "numpy<2" opencv-python-headless onnxruntime rtmlib pandas
!git clone --depth 1 https://github.com/Presto1001-cpu/arco.git /content/arco
%cd /content/arco/cello-tools
!/content/venv/bin/python -c "import mediapipe, cv2; print('ready', mediapipe.__version__, cv2.__version__)"

## 2. Upload your clips
Original video files, not screen recordings. Several files are fine.

In [ ]:
import os
from google.colab import files
os.makedirs('/content/clips', exist_ok=True)
up = files.upload()
for name, data in up.items():
    open('/content/clips/' + name, 'wb').write(data)
print(os.listdir('/content/clips'))

## 3. Run MediaPipe (baseline)
`--fps 30` analyses 30 frames per second. Use `--fps 60` for clips filmed at 60 fps or more.

In [ ]:
!/content/venv/bin/python handbench.py /content/clips/* --backend mediapipe --fps 30 --save-landmarks --out /content/out_mediapipe

## 4. Run RTMW (whole-body model)
RTMW finds the whole body first, so the player's body must be in the picture; a close-up of only a hand may not be found. The first run downloads its models. On the CPU this is slow, so this cell analyses at most 40 seconds at 15 frames per second. Remove `--max-seconds 40` for the full clip, or use a GPU runtime and replace `--device cpu` with `--device cuda` after running `!uv pip install --python /content/venv/bin/python onnxruntime-gpu`.

In [ ]:
!/content/venv/bin/python handbench.py /content/clips/* --backend rtmlib --rtm-mode balanced --device cpu --fps 15 --max-seconds 40 --save-landmarks --out /content/out_rtmw

## 5. Compare
One row per clip and speed band. "found" is the share of frames where a hand was found; "unstable" is how often the finger length jumped (a sign of guessing). A model that always finds a hand can still be wrong, which only hand labelling can show.

In [ ]:
import glob, json, pandas as pd
rows = []
for tag, folder in [('mediapipe', '/content/out_mediapipe'), ('rtmw', '/content/out_rtmw')]:
    for f in glob.glob(folder + '/*.json'):
        r = json.load(open(f))
        for b in r['summary']['bands']:
            rows.append(dict(clip=r['file'], model=tag, band=b['name'], frames=b['frames'], found=round(b['detected_pct']), unstable=round(b['unstable_pct']), confidence=round(b['mean_conf'], 2)))
df = pd.DataFrame(rows)
display(df.pivot_table(index=['clip', 'band'], columns='model', values=['found', 'unstable'], aggfunc='first'))

## 6. Download the results
Send the zip file back in the chat.

In [ ]:
!cd /content && zip -qr results.zip out_mediapipe out_rtmw
from google.colab import files
files.download('/content/results.zip')

## Next: HaMeR
HaMeR (3D hand mesh) is the strongest model on our list for hidden fingers, but it needs a separate setup and the MANO hand model, which requires a free registration at mano.is.tue.mpg.de and has its own licence terms. We add it only if MediaPipe and RTMW are not accurate enough after hand labelling.